In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

# ========== Load Files ==========
df1 = pd.read_csv("Normal_data.csv")
df2 = pd.read_csv("metasploitable-2.csv")
df3 = pd.read_csv("OVS.csv")

# ========== Merge ==========
df = pd.concat([df1, df2, df3], axis=0).reset_index(drop=True)

# ========== Clean ==========
df = df.drop_duplicates()
df = df.dropna()

# ========== Encode Categorical ==========
le = LabelEncoder()
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col])

# ========== Identify Label Column ==========
# Change if your label column name is different
label_col = "Label"

X = df.drop(label_col, axis=1)
y = df[label_col]

# ========== Normalize ==========
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# ========== Feature Extraction: PCA to 15 ==========
pca = PCA(n_components=15)
X_pca = pca.fit_transform(X_scaled)

# ========== Train Test Split ==========
X_train, X_test, y_train, y_test = train_test_split(
    X_pca, y, test_size=0.2, random_state=42, stratify=y
)

print("Final Shapes:")
print("Train:", X_train.shape, "Test:", X_test.shape)


Final Shapes:
Train: (275110, 15) Test: (68778, 15)


In [2]:
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model

inp = Input(shape=(X_train.shape[1],))   # 15 features
h1 = Dense(20, activation='relu')(inp)
h2 = Dense(12, activation='relu')(h1)    # deep features
out = Dense(X_train.shape[1], activation='linear')(h2)

mlp_model = Model(inp, out)
mlp_model.compile(optimizer='adam', loss='mse')


In [3]:
mlp_model.fit(X_train, X_train, epochs=25, batch_size=64, validation_split=0.1)


Epoch 1/25
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 18s 4ms/step - loss: 0.0191 - val_loss: 0.0032
Epoch 2/25
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 16s 4ms/step - loss: 0.0031 - val_loss: 0.0031
Epoch 3/25
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 18s 5ms/step - loss: 0.0031 - val_loss: 0.0031
Epoch 4/25
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - loss: 0.0031 - val_loss: 0.0031
Epoch 5/25
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - loss: 0.0031 - val_loss: 0.0031
Epoch 6/25
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - loss: 0.0031 - val_loss: 0.0031
Epoch 7/25
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - loss: 0.0031 - val_loss: 0.0031
Epoch 8/25
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - loss: 0.0031 - val_loss: 0.0031
Epoch 9/25
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - loss: 0.0031 - val_loss: 0.0031
Epoch 10/25
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - loss: 0.0031 - val_loss: 0.0031
Epoch 11/25
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - loss: 0.0031 - val_loss: 0.0031
Epoch 12/25
3869/38

In [4]:
mlp_encoder = Model(inputs=inp, outputs=h2)

X_train_mlp = mlp_encoder.predict(X_train)
X_test_mlp = mlp_encoder.predict(X_test)

print("MLP Feature Shape:", X_train_mlp.shape)   # (samples, 12)


8598/8598 ━━━━━━━━━━━━━━━━━━━━ 20s 2ms/step
2150/2150 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step
MLP Feature Shape: (275110, 12)


In [5]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

svm = SVC(kernel='rbf')
svm.fit(X_train_mlp, y_train)


SVC()

In [6]:
y_pred3 = svm.predict(X_test_mlp)

print("Hybrid Model 3: MLP + SVM")
print("Accuracy:", accuracy_score(y_test, y_pred3))
print(classification_report(y_test, y_pred3))


Hybrid Model 3: MLP + SVM
Accuracy: 0.9976445956555875
              precision    recall  f1-score   support

           0       0.85      0.93      0.89       281
           1       1.00      1.00      1.00        33
           2       1.00      1.00      1.00     14706
           3       1.00      1.00      1.00      9683
           4       1.00      0.99      0.99     10723
           5       1.00      1.00      1.00     13685
           6       1.00      1.00      1.00     19626
           7       0.00      0.00      0.00         3
           8       0.00      0.00      0.00        38

    accuracy                           1.00     68778
   macro avg       0.76      0.77      0.76     68778
weighted avg       1.00      1.00      1.00     68778



C:\Users\lenovo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\lenovo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\lenovo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
